# Bag of Words Model

Train and persist the BoW + Naive Bayes spam classifier.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import preprocess_many

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'SMSSpamCollection.txt'
MODEL_PATH = PROJECT_ROOT / 'models' / 'bow_nb.pkl'
VECTORIZER_PATH = PROJECT_ROOT / 'vectorizers' / 'bow.pkl'

messages = pd.read_csv(DATA_PATH, sep='\t', names=['label', 'message'])
corpus = preprocess_many(messages['message'].astype(str).tolist())
y = messages['label'].map({'ham': 0, 'spam': 1}).to_numpy()
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size=0.20, random_state=42, stratify=y)
cv = CountVectorizer(max_features=2500, ngram_range=(1, 2))
X_train_vec = cv.fit_transform(X_train).toarray()
X_test_vec = cv.transform(X_test).toarray()
bow_nb = MultinomialNB().fit(X_train_vec, y_train)
y_pred = bow_nb.predict(X_test_vec)
print('accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
VECTORIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bow_nb, MODEL_PATH)
joblib.dump(cv, VECTORIZER_PATH)
print(MODEL_PATH)
print(VECTORIZER_PATH)

accuracy: 0.9829596412556054
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.96      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

c:\Users\Yashesh Mehta\Desktop\Coding\SpamShield AI NLP-based Spam Detection System\models\bow_nb.pkl
c:\Users\Yashesh Mehta\Desktop\Coding\SpamShield AI NLP-based Spam Detection System\vectorizers\bow.pkl
